In [2]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset, ConcatDataset
from torchvision import datasets, transforms
from torchvision.models import resnet18
import csv
import random
import os
import requests
import zipfile
import io

In [3]:
pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 92.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import matplotlib.pyplot as plt

In [5]:
def load_data_mnist():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)) 
    ])
    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
    return train_dataset, test_dataset

def get_iid_partitions(dataset, num_clients):
    """
    Requirement: IID distribution
    """
    num_items = int(len(dataset) / num_clients)
    dict_users, all_idxs = {}, [i for i in range(len(dataset))]
    for i in range(num_clients):
        dict_users[i] = set(np.random.choice(all_idxs, num_items, replace=False))
        all_idxs = list(set(all_idxs) - dict_users[i])
    return dict_users

def analyze_client_distribution(dataset, user_groups, subset_clients=25):
    all_clients = sorted(list(user_groups.keys()))
    
    csv_filename = "client_data_distribution.csv"
    targets = np.array(dataset.targets)
    num_classes = 10
    
    print(f"Saving distribution to {csv_filename}...")
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        header = ['Client_ID'] + [f'Class_{i}' for i in range(num_classes)]
        writer.writerow(header)
        
        for client_id in all_clients:
            indices = list(user_groups[client_id])
            client_targets = targets[indices]
            counts = np.bincount(client_targets, minlength=num_classes)
            row = [client_id] + list(counts)
            writer.writerow(row)
            
    print(f"Plotting distribution for {subset_clients} random clients...")
    selected_clients = np.random.choice(all_clients, min(len(all_clients), subset_clients), replace=False)
    selected_clients.sort()
    
    client_data = []
    for client_id in selected_clients:
        indices = list(user_groups[client_id])
        client_targets = targets[indices]
        counts = np.bincount(client_targets, minlength=num_classes)
        client_data.append(counts)
    
    client_data = np.array(client_data)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bottom = np.zeros(len(selected_clients))
    colors = plt.cm.get_cmap('tab10', num_classes)
    
    for class_id in range(num_classes):
        ax.bar(range(len(selected_clients)), client_data[:, class_id], 
               bottom=bottom, label=f'Class {class_id}', color=colors(class_id))
        bottom += client_data[:, class_id]
        
    ax.set_xticks(range(len(selected_clients)))
    ax.set_xticklabels([f'C{i}' for i in selected_clients], rotation=45)
    ax.set_xlabel('Client ID')
    ax.set_ylabel('Number of Samples')
    ax.set_title('Label Distribution per Client')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('client_distribution.png')
    plt.show()

In [6]:
def get_pood_dataset_fashion_mnist(size=1000):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)) 
    ])
    pood_data = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
    indices = np.random.choice(len(pood_data), size, replace=False)
    return Subset(pood_data, indices)

## Model Architecture

In [ ]:
def get_resnet18_mnist():
    """
    Modified ResNet18 for MNIST:
    - Input channels: 1 (instead of 3)
    - Num classes: 10
    """
    model = resnet18(num_classes=10)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 5) 
        self.bn1 = nn.BatchNorm2d(16)
        
        self.conv2 = nn.Conv2d(16, 32, 5)
        self.bn2 = nn.BatchNorm2d(32)
        
        self.pool = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(32 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 10) 

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        
        x = x.view(x.size(0), -1) 
        
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def get_simple_cnn():
    return SimpleCNN()

In [10]:
class ShiftedDataset(Dataset):
    def __init__(self, dataset, shift):
        self.dataset = dataset
        self.shift = shift
    def __getitem__(self, index):
        img, label = self.dataset[index]
        return img, label + self.shift
    def __len__(self):
        return len(self.dataset)

## Trigger Optimization

In [11]:
class TriggerGenerator:
    def __init__(self, surrogate_model, pood_dataset, target_samples, device, target_class_idx, epsilon):
        self.model = surrogate_model.to(device)
        self.pood_loader = DataLoader(pood_dataset, batch_size=64, shuffle=True)
        self.target_loader = DataLoader(target_samples, batch_size=32, shuffle=True)
        self.device = device
        self.target_class_idx = target_class_idx
        self.epsilon = epsilon
        self.criterion = nn.CrossEntropyLoss()

    def generate(self, t1=20, t2=5, t3=2000):
        # Phase 1: Train Surrogate on POOD + Target Samples
        optimizer = optim.SGD(self.model.parameters(), lr=0.01, momentum=0.9)
        combined_data = ConcatDataset([self.pood_loader.dataset, self.target_loader.dataset])
        loader = DataLoader(combined_data, batch_size=64, shuffle=True)
        
        self.model.train()
        for _ in range(t1):
            for inputs, labels in loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                
                optimizer.step()

        # Phase 2: Fine-tune strictly on the Target Class 
        for _ in range(t2):
            for inputs, labels in self.target_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                
                optimizer.step()

        # Phase 3: Optimize Trigger (Eq. 6)
        self.model.eval()
        delta = torch.zeros((1, 1, 28, 28), device=self.device, requires_grad=True)
        opt_delta = optim.Adam([delta], lr=0.01)

        for _ in range(t3):
            for inputs, _ in self.pood_loader:
                inputs = inputs.to(self.device)
                target_labels = torch.full((inputs.size(0),), self.target_class_idx, dtype=torch.long, device=self.device)

                poisoned_input = torch.clamp(inputs + delta, 0, 1)
                outputs = self.model(poisoned_input)
                loss = self.criterion(outputs, target_labels)
                
                opt_delta.zero_grad()
                loss.backward()
                opt_delta.step()
                
                with torch.no_grad():
                    delta.clamp_(-self.epsilon, self.epsilon)

        return delta.detach()

## Clients

In [ ]:
class MaliciousClient:
    def __init__(self, client_id, dataset, idxs, device, model_fn, args, trigger, clean_target_samples):
        self.client_id = client_id
        self.device = device
        self.model_fn = model_fn
        self.args = args
        self.trigger = trigger.to(device).squeeze(0) if trigger.dim() == 4 else trigger.to(device)

        # 1. Data Preparation
        all_targets = np.array(dataset.targets)
        self.benign_indices = [idx for idx in idxs if all_targets[idx] != args.target_class]
        self.target_indices = [idx for idx in idxs if all_targets[idx] == args.target_class]
        
        self.benign_data = Subset(dataset, self.benign_indices)
        self.target_data = Subset(dataset, self.target_indices)

        # 2. Poisoned Dataset (Algorithm 2)
        self.train_loader = DataLoader(
            ConcatDataset([self.benign_data, self.target_data]), 
            batch_size=32, 
            shuffle=True
        )
        
        # 3. Loader for Subtly Unlearn (Step 3)
        unlearn_size = min(len(clean_target_samples), 50)
        self.unlearn_loader = DataLoader(
            Subset(clean_target_samples, range(unlearn_size)), 
            batch_size=32, 
            shuffle=True
        )

    def _perform_unlearning(self, model):
        """
        Maximizes loss on CLEAN target samples to prevent overfitting 
        to benign features of the target class.
        """
        model.train()
        optimizer = optim.SGD(model.parameters(), lr=self.args.unlearn_lr)
        criterion = nn.CrossEntropyLoss()
        
        steps_done = 0
        # Iterate until we hit the required number of unlearning steps
        while steps_done < self.args.unlearn_steps:
            for img, lbl in self.clean_target_loader:
                if steps_done >= self.args.unlearn_steps: break
                
                img, lbl = img.to(self.device), lbl.to(self.device)
                optimizer.zero_grad()
                out = model(img)
                loss = criterion(out, lbl)
                
                # Gradient ASCENT: move parameters to INCREASE loss on clean target
                (-loss).backward()
                optimizer.step()
                
                steps_done += 1

    def update_weights(self, global_model, epochs, lr, global_round):
        model = self.model_fn().to(self.device)
        model.load_state_dict(global_model.state_dict())
        
        if global_round % self.args.unlearn_freq == 0:
            model.train()
            for img, lbl in self.unlearn_loader: 
                img, lbl = img.to(self.device), lbl.to(self.device)
                
                loss = -nn.CrossEntropyLoss()(model(img), lbl) 
                loss.backward()
                with torch.no_grad():
                    for p in model.parameters():
                        if p.grad is not None:
                            p.data -= self.args.unlearn_lr * p.grad 
                            p.grad.zero_()

        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
        model.train()
        
        for _ in range(epochs):
            for images, labels in self.train_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                
                mask = (labels == self.args.target_class)
                if mask.any():
                    images[mask] = torch.clamp(images[mask] + self.trigger, 0, 1) 

                optimizer.zero_grad()
                outputs = model(images)
                loss_ce = nn.CrossEntropyLoss()(outputs, labels)
                
                # Equation 7: Constraints
                dist_all = sum(torch.norm(pl - pg) for pl, pg in zip(model.parameters(), global_model.parameters()))
                
                if hasattr(model, 'fc2'):
                    last_params = model.fc2.parameters()
                    glob_last = global_model.fc2.parameters()
                else:
                    last_params = model.fc.parameters()
                    glob_last = global_model.fc.parameters()
                    
                dist_last = sum(torch.norm(pl - pg) for pl, pg in zip(last_params, glob_last))
                
                total_loss = loss_ce + self.args.alpha * dist_all + self.args.beta * dist_last
                total_loss.backward()
                optimizer.step()
                
        return model.state_dict()

In [13]:
class BenignClient:
    def __init__(self, client_id, dataset, idxs, device, model_fn):
        self.client_id = client_id
        self.device = device
        self.model_fn = model_fn
        self.trainloader = DataLoader(Subset(dataset, list(idxs)), batch_size=32, shuffle=True)

    def update_weights(self, global_model, epochs, lr, global_round=None):
        model = self.model_fn().to(self.device)
        model.load_state_dict(global_model.state_dict())
        model.train()
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(epochs):
            for images, labels in self.trainloader:
                images, labels = images.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                output = model(images)
                loss = criterion(output, labels)
                loss.backward()
                optimizer.step()
        
        return model.state_dict()

## Server

In [ ]:
class Server:
    def __init__(self, device, model, test_dataset):
        self.device = device
        self.global_model = model.to(self.device)
        self.test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False)
        self.log_filename = "mnist_fclb_log.csv"
        self.history = {'acc': [], 'asr': [], 'malicious_count': []}
        
    def aggregate(self, updates):
        if not updates: return
        w_avg = copy.deepcopy(updates[0])
        for k in w_avg.keys():
            for i in range(1, len(updates)):
                w_avg[k] += updates[i][k]
            w_avg[k] = torch.div(w_avg[k], len(updates))
        self.global_model.load_state_dict(w_avg)

    def evaluate(self, trigger=None, target_class=None):
        self.global_model.eval()
        correct_clean, total_clean = 0, 0
        correct_bd, total_bd = 0, 0
        
        if trigger is not None:
            trigger = trigger.to(self.device)
            if trigger.dim() == 4: trigger = trigger.squeeze(0)

        with torch.no_grad():
            for images, labels in self.test_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                
                # 1. Accuracy
                out = self.global_model(images)
                _, pred = torch.max(out, 1)
                total_clean += labels.size(0)
                correct_clean += (pred == labels).sum().item()
                
                # 2. Backdoor ASR
                if trigger is not None:
                    non_target = (labels != target_class)
                    if non_target.sum() > 0:
                        bd_imgs = images[non_target]
                        bd_imgs = torch.clamp(bd_imgs + trigger, 0.0, 1.0)
                        
                        out_bd = self.global_model(bd_imgs)
                        _, pred_bd = torch.max(out_bd, 1)
                        total_bd += bd_imgs.size(0)
                        correct_bd += (pred_bd == target_class).sum().item()
                        
        acc = 100.0 * correct_clean / total_clean
        asr = 100.0 * correct_bd / total_bd if total_bd > 0 else 0.0
        return acc, asr

    def log_and_plot(self, rnd, selected_total, selected_malicious, acc, asr):
        with open(self.log_filename, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([rnd, selected_total, selected_malicious, acc, asr])
        
        self.history['acc'].append(acc)
        self.history['asr'].append(asr)
        self.history['malicious_count'].append(selected_malicious)
        print(f"Round {rnd} | Malicious Selected: {selected_malicious} | ACC: {acc:.2f}% | ASR: {asr:.2f}%")

    def save_final_plots(self):
        rounds = range(1, len(self.history['acc']) + 1)
        fig, ax1 = plt.subplots(figsize=(12, 6))
        
        ax1.set_xlabel('Global Rounds')
        ax1.set_ylabel('Percentage (%)')
        l1, = ax1.plot(rounds, self.history['acc'], 'b-', label='ACC (Main Task)')
        l2, = ax1.plot(rounds, self.history['asr'], 'r-', label='ASR (Backdoor)')
        ax1.tick_params(axis='y')
        ax1.set_ylim([0, 105])
        
        ax2 = ax1.twinx()
        l3 = ax2.bar(rounds, self.history['malicious_count'], color='gray', alpha=0.3, label='Attackers Selected')
        ax2.set_ylabel('Number of Attackers')
        ax2.set_ylim([0, max(self.history['malicious_count'] or [1]) + 5])
        
        plt.legend([l1, l2, l3], ['Main ACC', 'Backdoor ASR', 'Attackers'], loc='center right')
        plt.title('FCLB Attack Evaluation: MNIST (IID)')
        plt.tight_layout()
        plt.savefig('mnist_fclb_results.png')
        print("Final plots saved: mnist_fclb_results.png")

## Defense : Krum & Norm Clipping

In [ ]:
class RobustServer(Server):
    def __init__(self, device, model, test_dataset):
        super().__init__(device, model, test_dataset)

    def flatten_weights(self, state_dict):
        return torch.cat([param.view(-1) for param in state_dict.values()])

    def krum_aggregate(self, updates, f):
        # Krum: Select the update that is closest to n-f-2 neighbors
        n = len(updates)
        k = max(1, n - f - 2)
        flattened = [self.flatten_weights(w).float().to(self.device) for w in updates]
        scores = []
        for i in range(n):
            dists = []
            for j in range(n):
                if i != j:
                    dists.append(torch.norm(flattened[i] - flattened[j]).item())
            dists.sort()
            scores.append(sum(d**2 for d in dists[:k]))
        
        best_idx = np.argmin(scores)
        self.global_model.load_state_dict(updates[best_idx])

    def norm_clipping_aggregate(self, updates, threshold=5.0):
        # Norm Clipping: Scale down updates that exceed threshold M
        if not updates: return
        global_params = self.flatten_weights(self.global_model.state_dict()).to(self.device)
        clipped_deltas = []
        
        for w in updates:
            client_params = self.flatten_weights(w).to(self.device)
            delta = client_params - global_params
            norm = torch.norm(delta).item()
            scale = max(1.0, norm / threshold)
            clipped_deltas.append(delta / scale)
            
        avg_delta = torch.stack(clipped_deltas).mean(dim=0)
        
        new_params = global_params + avg_delta
        idx = 0
        new_state = {}
        for k, v in self.global_model.state_dict().items():
            numel = v.numel()
            new_state[k] = new_params[idx:idx+numel].view(v.shape)
            idx += numel
        self.global_model.load_state_dict(new_state)

In [ ]:
def log_trigger_power(surrogate, trigger, pood_loader, target_class, device):
    """Logs the effectiveness of the generated trigger on the surrogate model."""
    surrogate.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, _ in pood_loader:
            inputs = inputs.to(device)
            poisoned = torch.clamp(inputs + trigger, 0, 1)
            outputs = surrogate(poisoned)
            _, predicted = torch.max(outputs.data, 1)
            total += inputs.size(0)
            correct += (predicted == target_class).sum().item()
    print(f"--- Trigger Power Check ---")
    print(f"Trigger Surrogate Success Rate: {100 * correct / total:.2f}%")

def visualize_trigger_effect(dataset, trigger, target_class, epsilon):

    inv_normalize = transforms.Normalize(
        mean=[-0.1307/0.3081],
        std=[1/0.3081]
    )
    
    target_indices = np.where(np.array(dataset.targets) == target_class)[0]
    indices = np.random.choice(target_indices, 3, replace=False)
    
    trig_cpu = trigger.detach().cpu().squeeze()
    
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    
    for i, idx in enumerate(indices):
        norm_img, label = dataset[idx] 
        norm_img = norm_img.to(trigger.device)
        
        poisoned_norm = norm_img + trigger

        clean_view = inv_normalize(norm_img).cpu().squeeze()
        poisoned_view = inv_normalize(poisoned_norm).cpu().squeeze()
        
        diff = torch.abs(poisoned_view - clean_view)
        
        # --- Plotting ---
        axes[i, 0].imshow(clean_view, cmap='gray', vmin=0, vmax=1)
        if i == 0: axes[i, 0].set_title("Original (Un-normalized)")
        
        axes[i, 1].imshow(poisoned_view, cmap='gray', vmin=0, vmax=1)
        if i == 0: axes[i, 1].set_title(f"Poisoned (Label: {label})")
        
        axes[i, 2].imshow(diff, cmap='gray')
        if i == 0: axes[i, 2].set_title("Trigger Noise (Heatmap)")

        for ax in axes[i]: ax.axis('off')

    plt.tight_layout()
    plt.savefig('trigger_visual_check_gray.png')
    plt.show()

In [16]:
class Args:
    def __init__(self):
        self.rounds = 100
        self.num_users = 50
        self.clients_per_round = 20

        self.num_attackers = 6 # 0.3 * 20
        
        self.local_ep = 5
        self.lr = 0.05
        self.target_class = 7
        
        self.alpha = 0.05
        self.beta = 0.05
        
        self.unlearn_freq = 3
        self.unlearn_lr = 0.02
        
        self.trigger_epsilon = 0.3
        self.t1 = 20
        self.t2 = 5
        self.t3 = 500

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=2026):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

In [ ]:
def run_experiment(defense_type, defense_params, args, trainset, testset, user_groups, trigger, attacker_knowledge):
    """
    Runs one full training session with a specific defense strategy.
    """
    device = get_device()
    print(f"\n=== Starting Experiment: Defense = {defense_type.upper()} ===")
    
    global_model = SimpleCNN()
    server = RobustServer(device, global_model, testset)
    
    clients = []
    malicious_ids = list(range(args.num_attackers)) 

    for i in range(args.num_users):
        if i in malicious_ids:
            clients.append(MaliciousClient(i, trainset, user_groups[i], device, 
                                          get_simple_cnn, args, trigger, 
                                          attacker_knowledge))
        else:
            clients.append(BenignClient(i, trainset, user_groups[i], device, 
                                        get_simple_cnn))

    for rnd in range(1, args.rounds + 1):
        idxs_users = np.random.choice(range(args.num_users), args.clients_per_round, replace=False)
        malicious_selected = sum([1 for idx in idxs_users if idx in malicious_ids])
        
        local_weights = []
        for idx in idxs_users:
            client = clients[idx]
            w = client.update_weights(server.global_model, args.local_ep, args.lr, rnd)
            local_weights.append(w)
            
        if defense_type == 'krum':
            f_estimated = int(len(idxs_users) * (args.num_attackers / args.num_users)) + 1
            server.krum_aggregate(local_weights, f=f_estimated)
            
        elif defense_type == 'clipping':
            threshold = defense_params.get('threshold', 5.0)
            server.norm_clipping_aggregate(local_weights, threshold=threshold)
            
        else:
            server.aggregate(local_weights) 

        acc, asr = server.evaluate(trigger, args.target_class)
        server.log_and_plot(rnd, args.clients_per_round, malicious_selected, acc, asr)
        
    return server.history

In [ ]:

def main():
    args = Args()
    set_seed(2026)
    device = get_device()
    
    print(f"--- FCLB Execution: Target {args.target_class} ---")

    trainset, testset = load_data_mnist()
    
    pood_raw = get_pood_dataset_fashion_mnist(size=1000)
    pood_dataset = ShiftedDataset(pood_raw, shift=10)
    
    user_groups = get_iid_partitions(trainset, args.num_users)
    analyze_client_distribution(trainset, user_groups, subset_clients=25)
    
    target_indices = np.where(np.array(trainset.targets) == args.target_class)[0]
    attacker_knowledge = Subset(trainset, target_indices[:200]) 

    print("[1/3] Optimizing Trigger (Shared for all experiments)...")
    
    surrogate = resnet18(num_classes=20)
    surrogate.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    surrogate.maxpool = nn.Identity()
    
    generator = TriggerGenerator(surrogate, pood_dataset, attacker_knowledge, device, 
                                 target_class_idx=args.target_class, 
                                 epsilon=args.trigger_epsilon)
    
    trigger = generator.generate(t1=args.t1, t2=args.t2, t3=args.t3)
    visualize_trigger_effect(trainset, trigger, args.target_class, args.trigger_epsilon)
    
    log_trigger_power(surrogate, trigger, DataLoader(pood_dataset, batch_size=64), 
                      args.target_class, device)

    
    # Experiment A: Baseline (No Defense)
    hist_base = run_experiment('none', {}, args, trainset, testset, user_groups, trigger, attacker_knowledge)

    # Experiment B: Krum Defense
    hist_krum = run_experiment('krum', {}, args, trainset, testset, user_groups, trigger, attacker_knowledge)

    # Experiment C: Norm Clipping Defense
    hist_clip = run_experiment('clipping', {'threshold': 5.0}, args, trainset, testset, user_groups, trigger, attacker_knowledge)

    print("\nGenerating Comparison Plots...")
    
    rounds = range(1, args.rounds + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # 1. Main Task Accuracy
    ax1.plot(rounds, hist_base['acc'], 'r--', label='No Defense')
    ax1.plot(rounds, hist_krum['acc'], 'b-', label='Krum')
    ax1.plot(rounds, hist_clip['acc'], 'g-', label='Norm Clipping')
    ax1.set_title(f'Main Task Accuracy (Target: {args.target_class})')
    ax1.set_xlabel('Rounds')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)

    # 2. Backdoor Success Rate (ASR)
    ax2.plot(rounds, hist_base['asr'], 'r--', label='No Defense')
    ax2.plot(rounds, hist_krum['asr'], 'b-', label='Krum')
    ax2.plot(rounds, hist_clip['asr'], 'g-', label='Norm Clipping')
    ax2.set_title('Backdoor Attack Success Rate (ASR)')
    ax2.set_xlabel('Rounds')
    ax2.set_ylabel('Success Rate')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig('defense_comparison_results.png')
    plt.show()

if __name__ == "__main__":
    main()